# CIFAR-10 선형 분류

이 노트북은 CIFAR-10 데이터셋에 대해 선형 분류기를 구현하고 학습합니다. 각 이미지는 `32 x 32 x 3 = 3072`차원의 벡터로 변환되며, softmax cross-entropy 손실을 이용해 다중 클래스 분류를 수행합니다.

선형 분류기는 각 클래스마다 하나의 가중치 벡터를 학습하고, 입력 이미지 벡터와 가중치의 내적을 이용해 클래스별 점수를 계산합니다.

In [ ]:
# pathlib.Path는 운영체제에 상관없이 파일과 폴더 경로를 안전하게 다루기 위해 사용합니다.
# perf_counter는 학습에 걸린 시간을 비교적 정확하게 측정할 때 사용합니다.
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

# 그래프 배경에 격자를 넣어 loss와 accuracy 변화 추이를 더 쉽게 읽도록 설정합니다.
plt.style.use('seaborn-v0_8-whitegrid')

# 모든 결과 이미지는 results 폴더에 저장합니다.
# exist_ok=True는 폴더가 이미 있어도 오류가 나지 않게 합니다.
RESULT_DIR = Path('results')
RESULT_DIR.mkdir(exist_ok=True)

# CIFAR-10의 정답 라벨 번호(0~9)를 사람이 읽을 수 있는 클래스 이름으로 바꾸기 위한 목록입니다.
CIFAR10_LABELS = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

## 1. 데이터 불러오기 및 전처리

In [ ]:
# CIFAR-10 공식 Python 버전 데이터는 pickle 파일들이 tar.gz 압축 파일 안에 들어 있습니다.
# pickle은 CIFAR-10 배치 파일 읽기, tarfile은 압축 해제, urllib는 데이터셋 다운로드에 사용합니다.
import pickle
import tarfile
import urllib.request

# data 폴더는 원본 CIFAR-10 압축 파일과 압축 해제된 배치 파일을 보관하는 캐시 폴더입니다.
# 한 번 다운로드하면 이후 실행에서는 다시 다운로드하지 않습니다.
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CIFAR10_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CIFAR10_ARCHIVE = DATA_DIR / 'cifar-10-python.tar.gz'
CIFAR10_DIR = DATA_DIR / 'cifar-10-batches-py'

# 압축 해제된 CIFAR-10 폴더가 없을 때만 다운로드와 압축 해제를 수행합니다.
# 이렇게 하면 노트북을 반복 실행해도 불필요한 네트워크 요청을 줄일 수 있습니다.
if not CIFAR10_DIR.exists():
    if not CIFAR10_ARCHIVE.exists():
        print('CIFAR-10 다운로드 중...')
        urllib.request.urlretrieve(CIFAR10_URL, CIFAR10_ARCHIVE)
    with tarfile.open(CIFAR10_ARCHIVE, 'r:gz') as tar:
        tar.extractall(DATA_DIR)

def load_cifar_batch(path):
    """CIFAR-10 배치 파일 하나를 이미지 배열과 라벨 배열로 변환합니다."""
    with open(path, 'rb') as file:
        batch = pickle.load(file, encoding='latin1')
    # 원본 data는 (샘플 수, 3072) 형태입니다.
    # 3072 = 3채널(RGB) * 32높이 * 32너비이므로 다시 이미지 형태로 복원합니다.
    # transpose는 matplotlib이 바로 그릴 수 있는 (N, H, W, C) 형태로 축 순서를 바꿉니다.
    images = batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    labels = np.array(batch['labels'])
    return images, labels

# CIFAR-10 학습 데이터는 data_batch_1부터 data_batch_5까지 총 5개 파일로 나뉘어 있습니다.
# 각 배치를 읽은 뒤 concatenate로 하나의 학습 배열로 합칩니다.
train_images = []
train_labels = []
for batch_idx in range(1, 6):
    images, labels = load_cifar_batch(CIFAR10_DIR / f'data_batch_{batch_idx}')
    train_images.append(images)
    train_labels.append(labels)

# 최종 학습 데이터는 50,000장, 테스트 데이터는 10,000장으로 구성됩니다.
x_train = np.concatenate(train_images)
y_train = np.concatenate(train_labels)
x_test, y_test = load_cifar_batch(CIFAR10_DIR / 'test_batch')
y_train = y_train.ravel()
y_test = y_test.ravel()

print('학습 데이터:', x_train.shape, y_train.shape)
print('테스트 데이터:', x_test.shape, y_test.shape)

In [ ]:
# 전체 CIFAR-10을 모두 사용하면 실행 시간이 길어지므로 실습용으로 일부 샘플만 사용합니다.
# 선형 분류기의 동작과 성능 변화를 확인하기에는 이 정도 크기가 충분합니다.
TRAIN_LIMIT = 5000
TEST_LIMIT = 1000
RANDOM_SEED = 42

# RANDOM_SEED를 고정하면 매번 같은 샘플이 선택되어 결과를 재현할 수 있습니다.
rng = np.random.default_rng(RANDOM_SEED)
train_idx = rng.choice(len(x_train), size=TRAIN_LIMIT, replace=False)
test_idx = rng.choice(len(x_test), size=TEST_LIMIT, replace=False)

x_train_small = x_train[train_idx]
y_train_small = y_train[train_idx]
x_test_small = x_test[test_idx]
y_test_small = y_test[test_idx]

# 선형 분류기는 2차원 이미지가 아니라 1차원 특징 벡터를 입력으로 받습니다.
# 따라서 32x32x3 이미지를 길이 3072의 벡터로 펼치고, 픽셀값을 0~1 범위로 정규화합니다.
X_train = x_train_small.reshape(TRAIN_LIMIT, -1).astype(np.float32) / 255.0
X_test = x_test_small.reshape(TEST_LIMIT, -1).astype(np.float32) / 255.0

# 평균 이미지를 빼면 각 특징의 중심이 0에 가까워져 gradient descent가 더 안정적으로 진행됩니다.
mean_image = X_train.mean(axis=0, keepdims=True)
X_train_centered = X_train - mean_image
X_test_centered = X_test - mean_image

print('펼친 학습 데이터:', X_train_centered.shape)
print('펼친 테스트 데이터:', X_test_centered.shape)

In [ ]:
# 학습에 사용되는 샘플 일부를 먼저 확인하여 데이터가 올바르게 로드되었는지 검증합니다.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train_small[:10], y_train_small[:10]):
    ax.imshow(image)
    ax.set_title(CIFAR10_LABELS[label])
    ax.axis('off')
fig.suptitle('선형 분류에 사용한 CIFAR-10 샘플')
fig.tight_layout()
# 보고서나 README에서 재사용할 수 있도록 샘플 이미지를 파일로 저장합니다.
fig.savefig(RESULT_DIR / '6_cifar10_linear_samples.png', dpi=150)
plt.show()

## 2. Softmax 선형 분류기 구현

In [ ]:
def linear_scores(X, W, b):
    """입력 X에 대해 각 클래스의 선형 점수를 계산합니다."""
    # X: (샘플 수, 특징 수), W: (특징 수, 클래스 수), b: (클래스 수,)
    # 결과는 (샘플 수, 클래스 수)이며, 값이 클수록 해당 클래스일 가능성이 높다고 봅니다.
    return X @ W + b


def softmax_loss_and_grad(X, y, W, b, reg_strength):
    """Softmax cross-entropy 손실과 W, b에 대한 gradient를 함께 계산합니다."""
    num_samples = X.shape[0]
    scores = linear_scores(X, W, b)
    # softmax에서 exp 값이 너무 커지는 것을 막기 위해 각 샘플의 최대 점수를 빼줍니다.
    # 이 연산은 확률값을 바꾸지 않으면서 수치적으로 더 안정적인 계산을 가능하게 합니다.
    scores -= scores.max(axis=1, keepdims=True)

    exp_scores = np.exp(scores)
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    # 정답 클래스의 확률만 골라 -log를 취하면 cross-entropy loss가 됩니다.
    # 1e-12는 log(0)이 되는 극단적인 경우를 피하기 위한 작은 안정화 값입니다.
    correct_log_probs = -np.log(probabilities[np.arange(num_samples), y] + 1e-12)
    data_loss = correct_log_probs.mean()
    # L2 정규화는 가중치가 지나치게 커지는 것을 억제하여 과적합을 줄이는 역할을 합니다.
    reg_loss = 0.5 * reg_strength * np.sum(W * W)
    loss = data_loss + reg_loss

    # softmax + cross-entropy의 gradient는 예측 확률에서 정답 one-hot 벡터를 뺀 형태가 됩니다.
    dscores = probabilities
    dscores[np.arange(num_samples), y] -= 1.0
    dscores /= num_samples

    # chain rule에 따라 W와 b에 대한 gradient를 계산합니다.
    # W의 gradient에는 L2 정규화 항의 미분값(reg_strength * W)도 더합니다.
    dW = X.T @ dscores + reg_strength * W
    db = dscores.sum(axis=0)
    return loss, dW, db


def predict_linear_classifier(X, W, b):
    """각 샘플에서 점수가 가장 높은 클래스를 예측 라벨로 반환합니다."""
    return np.argmax(linear_scores(X, W, b), axis=1)


def accuracy_score(y_true, y_pred):
    """전체 샘플 중 예측이 정답과 일치한 비율을 계산합니다."""
    return np.mean(y_true == y_pred)


def train_linear_classifier(
    X_train, y_train, X_test, y_test,
    learning_rate=0.2, reg_strength=1e-4,
    num_iters=800, batch_size=256, random_seed=42,
):
    # 학습 과정에서도 seed를 고정해 mini-batch 선택과 초기 가중치를 재현 가능하게 만듭니다.
    rng = np.random.default_rng(random_seed)
    num_features = X_train.shape[1]
    num_classes = 10

    # W는 작은 난수로 초기화합니다. 너무 큰 값으로 시작하면 softmax 확률이 한쪽으로 치우칠 수 있습니다.
    # b는 클래스별 편향값이며 처음에는 모든 클래스에 동일하게 0을 부여합니다.
    W = 0.001 * rng.standard_normal((num_features, num_classes), dtype=np.float32)
    b = np.zeros(num_classes, dtype=np.float32)
    history = {'loss': [], 'train_acc': [], 'test_acc': [], 'iter': []}

    start_time = perf_counter()
    for step in range(1, num_iters + 1):
        # 매 반복마다 전체 데이터 대신 일부 mini-batch만 뽑아 gradient를 계산합니다.
        # 이 방식은 전체 데이터를 매번 쓰는 것보다 훨씬 빠르고, 실습 환경에서도 부담이 적습니다.
        batch_idx = rng.choice(len(X_train), size=batch_size, replace=False)
        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]

        loss, dW, db = softmax_loss_and_grad(X_batch, y_batch, W, b, reg_strength)
        # gradient descent 업데이트: 손실을 줄이는 방향으로 W와 b를 조금씩 이동시킵니다.
        # learning_rate는 한 번에 얼마나 크게 이동할지를 정하는 하이퍼파라미터입니다.
        W -= learning_rate * dW
        b -= learning_rate * db

        # 매 반복마다 정확도를 계산하면 느려지므로, 첫 반복/100회마다/마지막 반복에서만 기록합니다.
        if step == 1 or step % 100 == 0 or step == num_iters:
            train_pred = predict_linear_classifier(X_train, W, b)
            test_pred = predict_linear_classifier(X_test, W, b)
            train_acc = accuracy_score(y_train, train_pred)
            test_acc = accuracy_score(y_test, test_pred)

            history['loss'].append(loss)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['iter'].append(step)

            print(
                f'iter={step:4d}, loss={loss:.4f}, '
                f'train_acc={train_acc:.4f}, test_acc={test_acc:.4f}'
            )

    history['time_sec'] = perf_counter() - start_time
    return W, b, history

## 3. 모델 학습

In [ ]:
# 위에서 정의한 학습 함수를 실제 CIFAR-10 subset에 적용합니다.
# history에는 반복별 loss, train accuracy, test accuracy가 저장됩니다.
W, b, history = train_linear_classifier(
    X_train_centered,
    y_train_small,
    X_test_centered,
    y_test_small,
    learning_rate=0.2,
    reg_strength=1e-4,
    num_iters=800,
    batch_size=256,
    random_seed=RANDOM_SEED,
)

# 마지막으로 기록된 정확도를 최종 성능으로 사용합니다.
final_train_acc = history['train_acc'][-1]
final_test_acc = history['test_acc'][-1]
print(f'최종 학습 정확도: {final_train_acc:.4f}')
print(f'최종 테스트 정확도: {final_test_acc:.4f}')
print(f'학습 시간          : {history["time_sec"]:.2f}s')

In [ ]:
# 학습 과정에서 손실은 내려가는지, 정확도는 올라가는지 확인하기 위한 그래프입니다.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['iter'], history['loss'], marker='o')
axes[0].set_title('Softmax 손실')
axes[0].set_xlabel('반복 횟수')
axes[0].set_ylabel('손실')

axes[1].plot(history['iter'], history['train_acc'], marker='o', label='Train')
axes[1].plot(history['iter'], history['test_acc'], marker='o', label='Test')
axes[1].set_title('선형 분류기 정확도')
axes[1].set_xlabel('반복 횟수')
axes[1].set_ylabel('정확도')
axes[1].legend()

fig.tight_layout()
# 학습 곡선도 결과 이미지로 저장하여 보고서에 첨부할 수 있게 합니다.
fig.savefig(RESULT_DIR / '6_cifar10_linear_training_curves.png', dpi=150)
plt.show()

## 4. 학습률과 배치 크기의 Trade-off

학습률과 배치 크기는 학습 속도와 안정성을 결정하는 중요한 하이퍼파라미터입니다.

- 큰 `learning_rate`는 더 빠르게 학습할 수 있지만, 좋은 해를 지나치거나 loss를 불안정하게 만들 수 있습니다.
- 작은 `learning_rate`는 보통 더 안정적이지만, 정해진 반복 횟수 안에서 충분히 학습되지 않을 수 있습니다.
- 작은 `batch_size`는 업데이트에 노이즈가 많아 탐색에는 도움이 될 수 있지만, 정확도가 흔들릴 수 있습니다.
- 큰 `batch_size`는 더 안정적인 gradient 추정값을 제공하지만, 한 번의 업데이트 계산량이 커지고 항상 일반화 성능을 높이는 것은 아닙니다.

In [ ]:
# 이 sweep 실험은 learning rate, batch size, 정확도, 학습 시간 사이의 trade-off를 비교합니다.
# 실험 시간을 줄이고 비교를 명확하게 하기 위해 한 번에 하나의 변수만 바꾸고 나머지 설정은 고정합니다.
# 공정한 비교를 위해 random seed, 반복 횟수, 정규화 강도는 동일하게 유지합니다.
TRADEOFF_ITERS = 400
TRADEOFF_REG = 1e-4
DEFAULT_LR = 0.2
DEFAULT_BATCH_SIZE = 256

def run_tradeoff_case(learning_rate, batch_size):
    """주어진 learning rate와 batch size로 모델을 학습하고 요약 지표를 반환합니다."""
    W_tmp, b_tmp, history_tmp = train_linear_classifier(
        X_train_centered,
        y_train_small,
        X_test_centered,
        y_test_small,
        learning_rate=learning_rate,
        reg_strength=TRADEOFF_REG,
        num_iters=TRADEOFF_ITERS,
        batch_size=batch_size,
        random_seed=RANDOM_SEED,
    )
    return {
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'final_train_acc': history_tmp['train_acc'][-1],
        'final_test_acc': history_tmp['test_acc'][-1],
        'time_sec': history_tmp['time_sec'],
    }

# 1) Learning rate sweep: batch size를 256으로 고정하고 learning rate만 변경합니다.
# 작은 값은 보통 안정적이지만 느리고, 큰 값은 빠를 수 있지만 불안정할 수 있습니다.
learning_rates = [0.05, 0.1, 0.2, 0.5]
lr_tradeoff_results = []
for lr in learning_rates:
    print(f"\n[Learning rate sweep] lr={lr}, batch_size={DEFAULT_BATCH_SIZE}")
    lr_tradeoff_results.append(run_tradeoff_case(lr, DEFAULT_BATCH_SIZE))

# 2) Batch size sweep: learning rate를 0.2로 고정하고 batch size만 변경합니다.
# 작은 batch는 노이즈가 많고, 큰 batch는 더 안정적이지만 업데이트당 계산 비용이 커집니다.
batch_sizes = [64, 128, 256, 512]
batch_tradeoff_results = []
for batch_size in batch_sizes:
    print(f"\n[Batch size sweep] lr={DEFAULT_LR}, batch_size={batch_size}")
    batch_tradeoff_results.append(run_tradeoff_case(DEFAULT_LR, batch_size))

print("\nLearning rate trade-off 요약")
for result in lr_tradeoff_results:
    print(
        f"lr={result['learning_rate']:<4} | "
        f"test_acc={result['final_test_acc']:.4f} | "
        f"time={result['time_sec']:.2f}s"
    )

print("\nBatch size trade-off 요약")
for result in batch_tradeoff_results:
    print(
        f"batch_size={result['batch_size']:<3} | "
        f"test_acc={result['final_test_acc']:.4f} | "
        f"time={result['time_sec']:.2f}s"
    )

In [ ]:
# trade-off 결과를 정확도와 학습 시간 관점에서 함께 시각화합니다.
# 막대는 test accuracy를, 빨간 점선은 학습 시간을 나타냅니다.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lr_labels = [str(result['learning_rate']) for result in lr_tradeoff_results]
lr_accs = [result['final_test_acc'] for result in lr_tradeoff_results]
lr_times = [result['time_sec'] for result in lr_tradeoff_results]

batch_labels = [str(result['batch_size']) for result in batch_tradeoff_results]
batch_accs = [result['final_test_acc'] for result in batch_tradeoff_results]
batch_times = [result['time_sec'] for result in batch_tradeoff_results]

axes[0].bar(lr_labels, lr_accs, color='tab:blue', alpha=0.75, label='Test accuracy')
axes[0].set_title('Learning Rate Trade-off')
axes[0].set_xlabel('학습률')
axes[0].set_ylabel('테스트 정확도')
axes[0].set_ylim(0, max(lr_accs) + 0.05)
ax0_time = axes[0].twinx()
ax0_time.plot(lr_labels, lr_times, color='tab:red', marker='o', linestyle='--', label='Training time')
ax0_time.set_ylabel('시간(초)')

axes[1].bar(batch_labels, batch_accs, color='tab:green', alpha=0.75, label='Test accuracy')
axes[1].set_title('Batch Size Trade-off')
axes[1].set_xlabel('배치 크기')
axes[1].set_ylabel('테스트 정확도')
axes[1].set_ylim(0, max(batch_accs) + 0.05)
ax1_time = axes[1].twinx()
ax1_time.plot(batch_labels, batch_times, color='tab:red', marker='o', linestyle='--', label='Training time')
ax1_time.set_ylabel('시간(초)')

fig.tight_layout()
fig.savefig(RESULT_DIR / '6_cifar10_linear_tradeoffs.png', dpi=150)
plt.show()

## 5. 클래스별 가중치 시각화

선형 분류기의 가중치 벡터는 각 클래스에서 어떤 픽셀 패턴이 중요하게 작용하는지 보여줍니다. 아래 이미지는 클래스별 가중치 벡터를 다시 `32 x 32 x 3` 이미지 형태로 바꾸어 시각화한 것입니다.

In [ ]:
# W의 각 열은 하나의 클래스가 어떤 픽셀 패턴을 선호하는지 나타냅니다.
# 길이 3072의 가중치 벡터를 다시 32x32x3 이미지처럼 바꿔 시각화합니다.
weight_images = W.reshape(32, 32, 3, 10)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))

for class_idx, ax in enumerate(axes.ravel()):
    weight_img = weight_images[:, :, :, class_idx]
    # 가중치에는 음수와 양수가 섞여 있으므로, 보기 좋게 0~1 범위로 다시 스케일링합니다.
    weight_img = (weight_img - weight_img.min()) / (weight_img.max() - weight_img.min() + 1e-12)
    ax.imshow(weight_img)
    ax.set_title(CIFAR10_LABELS[class_idx])
    ax.axis('off')

fig.suptitle('학습된 선형 분류기 가중치')
fig.tight_layout()
fig.savefig(RESULT_DIR / '6_cifar10_linear_weights.png', dpi=150)
plt.show()

## 6. 선형 분류 결정 영역 시각화

원본 CIFAR-10 이미지는 3072차원이므로 결정 영역을 직접 시각화할 수 없습니다. 따라서 학습 데이터를 PCA로 2차원에 투영한 뒤, 그 2차원 평면에서 별도의 선형 분류기를 학습하여 결정 영역을 시각화합니다.

In [ ]:
# 원본 이미지는 3072차원이어서 결정 경계를 직접 그릴 수 없습니다.
# 따라서 일부 샘플을 2차원 PCA 공간으로 투영한 뒤, 그 평면에서 선형 분류기의 결정 영역을 시각화합니다.
REGION_TRAIN_LIMIT = 1200
REGION_TEST_LIMIT = 400
REGION_SCATTER_LIMIT = 400
GRID_SIZE = 160

X_region_source = X_train_centered[:REGION_TRAIN_LIMIT]
y_region = y_train_small[:REGION_TRAIN_LIMIT]
X_region_test_source = X_test_centered[:REGION_TEST_LIMIT]
y_region_test = y_test_small[:REGION_TEST_LIMIT]

# PCA도 평균을 뺀 데이터에서 수행해야 주성분이 데이터 분산 방향을 잘 나타냅니다.
region_mean = X_region_source.mean(axis=0, keepdims=True)
X_region_centered = X_region_source - region_mean
# SVD를 이용해 PCA의 주성분을 구합니다. vt의 앞 두 행이 분산이 가장 큰 두 방향입니다.
_, _, vt = np.linalg.svd(X_region_centered, full_matrices=False)
components = vt[:2]

# 고차원 이미지 벡터를 2개의 PCA 좌표로 투영합니다.
X_region_2d = X_region_centered @ components.T
X_region_test_2d = (X_region_test_source - region_mean) @ components.T

# 2차원 PCA 좌표에서 별도의 선형 분류기를 학습합니다.
# 이 모델은 시각화용이며, 위에서 학습한 3072차원 모델의 성능 평가와는 목적이 다릅니다.
W_region, b_region, region_history = train_linear_classifier(
    X_region_2d.astype(np.float32),
    y_region,
    X_region_test_2d.astype(np.float32),
    y_region_test,
    learning_rate=0.1,
    reg_strength=1e-4,
    num_iters=600,
    batch_size=256,
    random_seed=RANDOM_SEED,
)

# 2차원 평면 전체에 격자점을 만들고, 각 점을 어떤 클래스로 분류하는지 예측합니다.
# 이렇게 얻은 예측값을 색으로 칠하면 결정 영역(decision region)이 됩니다.
x_min, x_max = X_region_2d[:, 0].min() - 1.0, X_region_2d[:, 0].max() + 1.0
y_min, y_max = X_region_2d[:, 1].min() - 1.0, X_region_2d[:, 1].max() + 1.0
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, GRID_SIZE),
    np.linspace(y_min, y_max, GRID_SIZE),
)
grid_points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
region_pred = predict_linear_classifier(grid_points, W_region, b_region).reshape(xx.shape)

In [ ]:
# 배경색은 모델이 예측한 클래스 영역을, 점은 실제 학습 샘플의 위치와 라벨을 나타냅니다.
fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap('tab10')
# 클래스 라벨이 0~9의 정수이므로 각 클래스가 구분되어 칠해지도록 경계값을 0.5 간격으로 둡니다.
levels = np.arange(11) - 0.5

ax.contourf(xx, yy, region_pred, levels=levels, cmap=cmap, alpha=0.28)
scatter = ax.scatter(
    X_region_2d[:REGION_SCATTER_LIMIT, 0],
    X_region_2d[:REGION_SCATTER_LIMIT, 1],
    c=y_region[:REGION_SCATTER_LIMIT],
    cmap=cmap,
    s=18,
    edgecolors='k',
    linewidths=0.25,
)
ax.set_title('2D PCA 투영에서의 선형 분류 결정 영역')
ax.set_xlabel('PCA 성분 1')
ax.set_ylabel('PCA 성분 2')

cbar = fig.colorbar(scatter, ax=ax, ticks=np.arange(10), fraction=0.035, pad=0.02)
cbar.ax.set_yticklabels(CIFAR10_LABELS)
fig.tight_layout()
# 결정 영역 시각화도 결과 이미지로 저장합니다.
fig.savefig(RESULT_DIR / '6_cifar10_linear_decision_regions.png', dpi=150)
plt.show()

## 7. 분석

- 선형 분류기는 입력 이미지 벡터와 클래스별 가중치 벡터의 내적을 이용해 클래스 점수를 계산합니다.
- softmax 함수는 클래스 점수를 확률처럼 해석할 수 있는 값으로 변환하고, cross-entropy 손실은 정답 클래스의 확률이 커지도록 학습을 유도합니다.
- KNN은 학습 데이터를 저장해 두었다가 테스트 시점에 가까운 예시를 찾는 방식입니다. 반면 선형 분류기는 학습 과정에서 `W`와 `b`를 학습하고, 테스트 시점에는 한 번의 행렬 곱으로 예측합니다.
- CIFAR-10처럼 복잡한 이미지에서는 단순한 선형 경계만으로 모든 클래스를 정확히 분리하기 어렵습니다. 하지만 학습 곡선과 가중치 이미지를 통해 분류기가 어떤 패턴을 학습하는지 확인할 수 있습니다.
- 결정 영역 그래프는 2차원 PCA 공간에서 따로 학습한 선형 분류기에서 나온 것이므로 원래의 3072차원 분류 결과와 완전히 같지는 않지만, 선형 결정 경계가 공간을 어떻게 나누는지 직관적으로 보여줍니다.
- learning rate 실험에서는 값이 너무 커질수록 업데이트가 과해져 테스트 정확도가 낮아질 수 있습니다.
- batch size 실험에서는 작은 batch가 항상 빠르고 좋은 것은 아니며, 더 안정적인 gradient를 주는 큰 batch가 더 나은 정확도를 보일 수도 있습니다.